# EDA — Credit Card Fraud Data

Exploratory data analysis for `creditcard.csv` — anonymized PCA features V1–V28 plus Amount and Time.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

cc_df = pd.read_csv('../data/raw/creditcard.csv')
print('Shape:', cc_df.shape)
cc_df.head()

## 1. Data Quality

In [ ]:
cc_df.info()
print('\nMissing values:')
print(cc_df.isnull().sum().sum(), 'total')
print('\nDuplicates:', cc_df.duplicated().sum())

# Drop duplicates if any
cc_df = cc_df.drop_duplicates().reset_index(drop=True)
print('Shape after dedup:', cc_df.shape)

## 2. Class Imbalance

In [ ]:
class_counts = cc_df['Class'].value_counts()
print(class_counts)
print(f'\nFraud rate: {class_counts[1] / len(cc_df) * 100:.4f}%')

fig, ax = plt.subplots(figsize=(5, 4))
class_counts.plot(kind='bar', color=['steelblue', 'tomato'], ax=ax)
ax.set_xticklabels(['Legitimate (0)', 'Fraud (1)'], rotation=0)
ax.set_title('Class Distribution — creditcard')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 3. Amount and Time Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cc_df['Amount'].hist(bins=100, ax=axes[0], color='steelblue')
axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount ($)')

cc_df['Time'].hist(bins=100, ax=axes[1], color='steelblue')
axes[1].set_title('Time Distribution (seconds since first tx)')
axes[1].set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

In [ ]:
# Amount and Time vs Class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cc_df.boxplot(column='Amount', by='Class', ax=axes[0])
axes[0].set_title('Amount by Class')
axes[0].set_xlabel('Class')

cc_df.boxplot(column='Time', by='Class', ax=axes[1])
axes[1].set_title('Time by Class')
axes[1].set_xlabel('Class')

plt.suptitle('')
plt.tight_layout()
plt.show()

## 4. PCA Feature Correlations with Class

In [ ]:
v_cols = [f'V{i}' for i in range(1, 29)]
correlations = cc_df[v_cols + ['Amount', 'Time']].corrwith(cc_df['Class']).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
correlations.plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Correlation of Features with Class (Fraud)')
ax.set_xlabel('Pearson Correlation')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print('\nTop positive correlators (fraud signal):')
print(correlations.tail(5))
print('\nTop negative correlators:')
print(correlations.head(5))

## 5. Save Cleaned Dataset

In [ ]:
cc_df.to_csv('../data/processed/creditcard_clean.csv', index=False)
print('Saved to data/processed/creditcard_clean.csv')
print('Shape:', cc_df.shape)